<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/OB3%EC%A1%B0_%ED%94%84%EB%A1%9C%EC%A0%9D%ED%8A%B82_%EC%9D%B4%EB%82%98%EB%A0%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. 환경 설치 - 학습 라이브러리 준

In [ ]:
!pip -q install "transformers>=4.44,<5" "datasets>=2.19,<4" "accelerate>=0.33" sentencepiece protobuf scikit-learn pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


# 1. 설정 - T4에 맞춰 배치 크기와 학습량 지정

In [ ]:
import os, json, re, gc, random, collections, pathlib
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments, DataCollatorForTokenClassification
from datasets import Dataset
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — T4 설정 권장')
# Colab: /content/train.json, /content/test.json / Kaggle: 자동 탐색
ROOT='/content'
if not os.path.exists(f'{ROOT}/train.json'):
    matches=list(pathlib.Path('/kaggle/input').glob('**/train.json')) if os.path.exists('/kaggle/input') else []
    if matches: ROOT=str(matches[0].parent)
TRAIN=f'{ROOT}/train.json'; TEST=f'{ROOT}/test.json'
OUTPUT='/content/pii_output' if os.path.exists('/content') else '/kaggle/working/pii_output'
os.makedirs(OUTPUT,exist_ok=True)
MODEL_A='microsoft/deberta-v3-base'
MODEL_B='microsoft/deberta-v3-small' # 다른 구조로 다양성 확보
RUN_SECOND_MODEL=False  # 첫 모델 완료 후 True로 바꿔 비교
EPOCHS=2; LR=2e-5; CHUNK_WORDS=220; CHUNK_STEP=180
BATCH=4; ACCUM=4; MAX_LENGTH=384
assert os.path.exists(TRAIN) and os.path.exists(TEST), 'train.json, test.json을 /content에 업로드하세요.'
train=json.load(open(TRAIN,encoding='utf-8')); test=json.load(open(TEST,encoding='utf-8'))
LABELS=['O']+sorted({y for d in train for y in d['labels'] if y!='O'})
L2I={x:i for i,x in enumerate(LABELS)}; I2L={i:x for x,i in L2I.items()}
print('train',len(train),'test',len(test),'labels',LABELS)

GPU: Tesla T4
train 6807 test 10 labels ['O', 'B-EMAIL', 'B-ID_NUM', 'B-NAME_STUDENT', 'B-PHONE_NUM', 'B-STREET_ADDRESS', 'B-URL_PERSONAL', 'B-USERNAME', 'I-ID_NUM', 'I-NAME_STUDENT', 'I-PHONE_NUM', 'I-STREET_ADDRESS', 'I-URL_PERSONAL']


#2. EDA - 라벨 희소성 , 문서 길이, 위치 확인

In [ ]:
cnt=collections.Counter(y for d in train for y in d['labels'])
print(pd.DataFrame(cnt.most_common(),columns=['label','tokens']).to_string(index=False))
print('PII docs:',sum(any(y!='O' for y in d['labels']) for d in train))
print('Token length percentiles:',np.percentile([len(d['tokens']) for d in train],[50,90,95,99,100]))
print('PII token ratio:',round(1-cnt['O']/sum(cnt.values()),6))
print('BIO counts:',sum(v for k,v in cnt.items() if k.startswith('B-')),sum(v for k,v in cnt.items() if k.startswith('I-')))

           label  tokens
               O 4989794
  B-NAME_STUDENT    1365
  I-NAME_STUDENT    1096
  B-URL_PERSONAL     110
        B-ID_NUM      78
         B-EMAIL      39
I-STREET_ADDRESS      20
     I-PHONE_NUM      15
      B-USERNAME       6
     B-PHONE_NUM       6
B-STREET_ADDRESS       2
  I-URL_PERSONAL       1
        I-ID_NUM       1
PII docs: 945
Token length percentiles: [ 693. 1123. 1302. 1733. 3298.]
PII token ratio: 0.000549
BIO counts: 1606 1133


# 3. 검증 분할 - PII 포함 문서 비율 유지 (모든 모델에 동일 분할)

In [ ]:
flags=[int(any(y!='O' for y in d['labels'])) for d in train]
tr_idx,va_idx=train_test_split(np.arange(len(train)),test_size=.18,random_state=SEED,stratify=flags)
tr_docs=[train[i] for i in tr_idx]; va_docs=[train[i] for i in va_idx]
print('train docs',len(tr_docs),'valid docs',len(va_docs))

train docs 5581 valid docs 1226


# 4. 토큰 정렬 - 원본 spaCy token ID 유지, 겹치는 창으로 긴 글 보장

In [ ]:
def chunks(docs, labeled=False):
    rows=[]
    for d in docs:
        tokens=d['tokens']
        for start in range(0,len(tokens),CHUNK_STEP):
            end=min(start+CHUNK_WORDS,len(tokens))
            # 공백 전용 spaCy 토큰도 word_ids에서 유지되도록 대체
            words=[w if w.strip() else '[unused0]' for w in tokens[start:end]]
            rows.append({'doc':d['document'],'start':start,'words':words,
                         'gold':d['labels'][start:end] if labeled else None})
            if end==len(tokens):break
    return rows

def encode(rows, tokenizer, labeled=False):
    features=[]; maps=[]
    for r in rows:
        enc=tokenizer(r['words'],is_split_into_words=True,truncation=True,max_length=MAX_LENGTH,
                      return_overflowing_tokens=True,stride=32)
        for j in range(len(enc['input_ids'])):
            ids=enc.word_ids(batch_index=j)
            feat={'input_ids':enc['input_ids'][j],'attention_mask':enc['attention_mask'][j]}
            labs=[]; wmap=[]; prev=None
            for w in ids:
                first=w is not None and w!=prev
                idx=r['start']+w if w is not None else -1
                wmap.append(idx if first else -1)
                if labeled:
                    labs.append(L2I[r['gold'][w]] if first and r['words'][w]!='[unused0]' else -100)
                prev=w
            if labeled:feat['labels']=labs
            features.append(feat); maps.append((r['doc'],wmap))
    return Dataset.from_list(features),maps

def f5_score(gold,pred):
    tp=fp=fn=0
    for d in gold:
        g=d['labels']; p=pred[d['document']]
        for a,b in zip(g,p):
            if a==b and a!='O':tp+=1
            elif a!=b:
                if b!='O':fp+=1
                if a!='O':fn+=1
    return 26*tp/(26*tp+25*fn+fp) if tp+fp+fn else 0.,(tp,fp,fn)

def probs_to_labels(probs,threshold=0.25):
    out={}
    for doc,p in probs.items():
        pos=p[:,1:]; arg=pos.argmax(axis=1)+1; best=pos.max(axis=1)
        out[doc]=[I2L[int(k)] if s>=threshold else 'O' for k,s in zip(arg,best)]
    return out

def infer(model,tokenizer,docs):
    rows=chunks(docs); ds,maps=encode(rows,tokenizer,False)
    collator=DataCollatorForTokenClassification(tokenizer)
    from torch.utils.data import DataLoader
    loader=DataLoader(ds,batch_size=8,collate_fn=collator)
    sums={d['document']:np.zeros((len(d['tokens']),len(LABELS)),np.float32) for d in docs}
    counts={d['document']:np.zeros(len(d['tokens']),np.int32) for d in docs}
    model.eval(); model.to('cuda' if torch.cuda.is_available() else 'cpu')
    k=0
    with torch.inference_mode():
        for batch in loader:
            ids=batch['input_ids'].to(model.device); mask=batch['attention_mask'].to(model.device)
            p=model(input_ids=ids,attention_mask=mask).logits.float().softmax(-1).cpu().numpy()
            for i in range(len(p)):
                doc,wmap=maps[k]; k+=1
                for j,w in enumerate(wmap):
                    if w>=0 and j<len(p[i]):
                        sums[doc][w]+=p[i,j];counts[doc][w]+=1
    for doc in sums:
        sums[doc]/=np.maximum(counts[doc][:,None],1)
        missing=counts[doc]==0
        if missing.any():sums[doc][missing,0]=1.
    return sums

def tune(probs,docs):
    results=[]
    for th in [0.08,0.12,0.16,0.20,0.25,0.30,0.35,0.40,0.50,0.60]:
        score,detail=f5_score(docs,probs_to_labels(probs,th));results.append((score,th,detail))
    return sorted(results,reverse=True)[0],sorted(results,reverse=True)

# 5. 모델 학습 - DeBERTa Token Classification (T4 혼합정밀도)

데이터 형식 변환 -> 모델 학습 -> Validation F5 최적

In [ ]:
import inspect
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)


def fit_model(model_name, run_name):

    # 1. Tokenizer
    print("\n[1/5] Tokenizer 로딩")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=True
    )

    assert tokenizer.is_fast, "Fast tokenizer가 필요합니다."

    # 2. 학습 데이터 생성
    print("\n[2/5] 학습 데이터 생성")

    tr_ds, train_maps = encode(
        chunks(tr_docs, True),
        tokenizer,
        True
    )

    print("학습 샘플 수:", len(tr_ds))
    print("첫 번째 샘플 형식:", type(tr_ds[0]))
    print("첫 번째 샘플 항목:", tr_ds[0].keys())

    assert len(tr_ds) > 0, "학습 데이터가 비어 있습니다."

    required_keys = {
        "input_ids",
        "attention_mask",
        "labels"
    }

    assert required_keys.issubset(tr_ds[0].keys()), (
        "학습 데이터에 필요한 항목이 없습니다."
    )



    # 3. DeBERTa 모델 생성
    # FP32로 유지하여 gradient scaling 오류 방지
    print("\n[3/5] DeBERTa 모델 로딩")

    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(LABELS),
        id2label=I2L,
        label2id=L2I,
        ignore_mismatched_sizes=True
    )

    model = model.float()

    print(
        "모델 dtype:",
        next(model.parameters()).dtype
    )

    # 4. TrainingArguments
    # transformers 버전 호환성 처리
    print("\n[4/5] 학습 설정")

    args = dict(
        output_dir=f"{OUTPUT}/{run_name}",
        learning_rate=LR,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=ACCUM,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        warmup_ratio=0.08,

        # FP16 gradient 오류 방지
        fp16=False,
        bf16=False,

        logging_steps=50,
        save_strategy="no",
        report_to="none",
        seed=SEED,

        # multiprocessing 관련 오류 방지
        dataloader_num_workers=0
    )

    supported = inspect.signature(
        TrainingArguments.__init__
    ).parameters

    # warmup_ratio 미지원 버전 대응
    if "warmup_ratio" not in supported:
        args.pop("warmup_ratio", None)

        if "warmup_steps" in supported:
            args["warmup_steps"] = 100

    # 설치된 버전에서 지원하는 인자만 사용
    unsupported = [
        k for k in args if k not in supported
    ]

    if unsupported:
        print("미지원 인자 제외:", unsupported)

    args = {
        k: v
        for k, v in args.items()
        if k in supported
    }

    ta = TrainingArguments(**args)

    print("FP16 설정:", ta.fp16)
    print("모델 dtype:", next(model.parameters()).dtype)
    print("Batch size:", BATCH)
    print("Gradient accumulation:", ACCUM)
    print("Epochs:", EPOCHS)



    # 5. Trainer 생성 및 학습
    print("\n[5/5] 모델 학습 시작")

    trainer = Trainer(
        model=model,
        args=ta,
        train_dataset=tr_ds,
        data_collator=DataCollatorForTokenClassification(
            tokenizer=tokenizer
        )
    )

    trainer.train()

    print("\n모델 학습 완료!")



    # 6. Validation 예측
    print("\nValidation 예측 시작")

    model.eval()

    val = infer(
        model,
        tokenizer,
        va_docs
    )

    # 7. F5 Threshold 최적화
    print("\nF5 Threshold 최적화")

    best, table = tune(
        val,
        va_docs
    )

    print("\n" + "=" * 60)
    print("모델:", run_name)
    print("BEST F5, threshold, (TP, FP, FN):")
    print(best)
    print("\nTop 5 thresholds:")
    print(table[:5])
    print("=" * 60)

    return model, tokenizer, val, best


# DeBERTa-v3-base 학습 실행

model_a, tok_a, val_a, best_a = fit_model(
    MODEL_A,
    "deberta_base"
)




[1/5] Tokenizer 로딩

[2/5] 학습 데이터 생성
학습 샘플 수: 24934
첫 번째 샘플 형식: <class 'dict'>
첫 번째 샘플 항목: dict_keys(['input_ids', 'attention_mask', 'labels'])

[3/5] DeBERTa 모델 로딩


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task

모델 dtype: torch.float32

[4/5] 학습 설정
FP16 설정: False
모델 dtype: torch.float32
Batch size: 4
Gradient accumulation: 4
Epochs: 2

[5/5] 모델 학습 시작


Step,Training Loss
50,6.308328
100,0.019513
150,0.009788
200,0.005002
250,0.002317
300,0.003553
350,0.003438
400,0.002411
450,0.003832
500,0.003111



모델 학습 완료!

Validation 예측 시작

F5 Threshold 최적화

모델: deberta_base
BEST F5, threshold, (TP, FP, FN):
(0.9150943396226415, 0.08, (485, 220, 38))

Top 5 thresholds:
[(0.9150943396226415, 0.08, (485, 220, 38)), (0.9139072847682119, 0.12, (483, 183, 40)), (0.911183970856102, 0.16, (481, 169, 42)), (0.9093165887850467, 0.2, (479, 142, 44)), (0.8976009362200117, 0.25, (472, 125, 51))]


# 6. 두 번째 모델 - 필요할 때만 학습, 단일 모델과 앙상블 비교

In [ ]:
val_b=None; model_b=None;tok_b=None
if RUN_SECOND_MODEL:
    model_b,tok_b,val_b,best_b=fit_model(MODEL_B,'deberta_small')
    for w in [0.25,0.50,0.75]:
        ensemble={k:w*val_a[k]+(1-w)*val_b[k] for k in val_a}
        print('A weight',w,'best',tune(ensemble,va_docs)[0])
else:
    print('모델 A만 사용. RUN_SECOND_MODEL=True로 바꾸고 6번 셀 재실행하면 비교 가능.')

모델 A만 사용. RUN_SECOND_MODEL=True로 바꾸고 6번 셀 재실행하면 비교 가능.


# 7. 후처리 - 문서 내 반복 이름 보완, 검증 점수로 채택 여부 결정

In [ ]:
def repeat_name_postprocess(pred,docs):
    out={k:v.copy() for k,v in pred.items()}
    for d in docs:
        doc=d['document'];tokens=d['tokens'];labs=out[doc]
        # 확실한 2토큰 이상 NAME 엔티티만 동일 문서 내 정확히 일치하는 구절에 전파
        names=set()
        for i in range(len(labs)-1):
            if labs[i]=='B-NAME_STUDENT' and labs[i+1]=='I-NAME_STUDENT':
                if all(t.isalpha() and len(t)>1 for t in tokens[i:i+2]):
                    names.add((tokens[i].lower(),tokens[i+1].lower()))
        for i in range(len(labs)-1):
            pair=(tokens[i].lower(),tokens[i+1].lower())
            if pair in names and labs[i]=='O' and labs[i+1]=='O':
                labs[i]='B-NAME_STUDENT';labs[i+1]='I-NAME_STUDENT'
    return out

candidates=[('A',val_a)]
if val_b is not None:
    candidates.append(('B',val_b))
    for w in [0.25,0.50,0.75]:
        candidates.append((f'ensemble_A_{w}',{k:w*val_a[k]+(1-w)*val_b[k] for k in val_a}))
results=[]
for name,p in candidates:
    best,_=tune(p,va_docs)
    for post in [False,True]:
        labels=probs_to_labels(p,best[1]);labels=repeat_name_postprocess(labels,va_docs) if post else labels
        score,detail=f5_score(va_docs,labels)
        results.append((score,name,best[1],post,detail))
results.sort(reverse=True)
print(pd.DataFrame(results,columns=['F5','model','threshold','repeat_name','TP_FP_FN']).to_string(index=False))
selected=results[0];print('SELECTED:',selected)

      F5 model  threshold  repeat_name       TP_FP_FN
0.915094     A       0.08        False (485, 220, 38)
0.914298     A       0.08         True (485, 232, 38)
SELECTED: (0.9150943396226415, 'A', 0.08, False, (485, 220, 38))


# 8. 모델 저장

In [ ]:
# 현재 학습 완료된 모델 A 저장
SAVE_PATH = "/content/pii_deberta_base"

model_a.save_pretrained(SAVE_PATH)
tok_a.save_pretrained(SAVE_PATH)

print("모델 저장 완료:", SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

모델 저장 완료: /content/pii_deberta_base


In [ ]:
if isinstance(val_a, dict):
    print("val_a keys:", list(val_a.keys())[:10])
    first = next(iter(val_a.values()))
else:
    first = val_a[0]

print("first item type:", type(first))

if isinstance(first, dict):
    print("first item keys:", list(first.keys()))
    for k, v in first.items():
        print(k, type(v), getattr(v, "shape", None))
else:
    print("first item preview:", str(first)[:500])

print("best_a:", best_a)

val_a keys: [15612, 13471, 17451, 17621, 9670, 21230, 19680, 10555, 17054, 14426]
first item type: <class 'numpy.ndarray'>
first item preview: [[9.99996424e-01 3.61885299e-07 5.69292922e-07 ... 9.77161605e-08
  5.45955970e-07 1.56219102e-07]
 [9.99997020e-01 2.64996260e-07 5.21483571e-07 ... 7.72377007e-08
  5.56739167e-07 1.13559885e-07]
 [9.99997020e-01 2.83320787e-07 5.00459521e-07 ... 8.41799519e-08
  4.49523327e-07 9.70242340e-08]
 ...
 [9.99996901e-01 3.00521293e-07 4.47518886e-07 ... 7.61560557e-08
  4.30113317e-07 1.23018054e-07]
 [9.99996424e-01 3.78570661e-07 6.22229550e-07 ... 1.18338200e-07
  4.19759061e-07 9.54944568e-08]

best_a: (0.9150943396226415, 0.08, (485, 220, 38))


# 10. Threshold 세부 탐

In [ ]:
# 10. 모델 A Threshold 세부 탐색
# 재학습 없이 기존 Validation 예측 결과 사용
# 실제 threshold별 F5를 다시 계산하지는 않고, 확률 분포와 tune() 함수의 인자를 확인하는 진단 코드

import numpy as np
import pandas as pd
import inspect

print("현재 BEST:", best_a)
print("tune 함수 인자:", inspect.signature(tune))

# 기존 tune 함수의 구현을 확인
print("\n현재 Validation 문서 수:", len(va_docs))
print("예측 문서 수:", len(val_a))

# 각 라벨별 예측 확률 분포 확인
all_max_probs = []

for doc_id, probs in val_a.items():
    probs = np.asarray(probs)

    # O를 제외한 PII 라벨 중 최대 확률
    pii_probs = probs[:, 1:].max(axis=1)
    all_max_probs.append(pii_probs)

all_max_probs = np.concatenate(all_max_probs)

print("\nPII 최대 예측 확률 분포:")
print(
    pd.Series(all_max_probs).quantile(
        [0, 0.5, 0.9, 0.95, 0.99, 0.999, 1.0]
    )
)

# 기존 tune 함수가 threshold 목록을 외부에서
# 받는지 확인하기 위한 정보 출력
print("\n기존 threshold 최적값:", best_a[1])
print("다음 단계: tune 함수 구현에 맞춰 세부 탐색")


현재 BEST: (0.9150943396226415, 0.08, (485, 220, 38))
tune 함수 인자: (probs, docs)

현재 Validation 문서 수: 1226
예측 문서 수: 1226

PII 최대 예측 확률 분포:
0.000    3.248451e-07
0.500    4.559269e-07
0.900    8.334803e-07
0.950    2.701567e-06
0.990    2.671955e-05
0.999    1.310186e-02
1.000    9.978277e-01
dtype: float64

기존 threshold 최적값: 0.08
다음 단계: tune 함수 구현에 맞춰 세부 탐색


# 11. F5 최적화 함수 구조 확인

In [ ]:
import inspect

print("===== TUNE FUNCTION =====")
print(inspect.getsource(tune))

print("\n===== AVAILABLE FUNCTIONS =====")
for name in ["score", "decode", "predict", "infer"]:
    obj = globals().get(name)
    if callable(obj):
        print(f"\n{name}: {inspect.signature(obj)}")

===== TUNE FUNCTION =====
def tune(probs,docs):
    results=[]
    for th in [0.08,0.12,0.16,0.20,0.25,0.30,0.35,0.40,0.50,0.60]:
        score,detail=f5_score(docs,probs_to_labels(probs,th));results.append((score,th,detail))
    return sorted(results,reverse=True)[0],sorted(results,reverse=True)


===== AVAILABLE FUNCTIONS =====

infer: (model, tokenizer, docs)


- 현재 tune() 함수는 Threshold 10개만 비교

- 현재 최고 점수인 F5 = 0.915094, Threshold = 0.08은 전체 범위에서 최적이 아닐 수도 있음.

# 12. Threshold 정밀 최적화

In [ ]:
# 12. Threshold 정밀 최적화
# 기존 모델 A의 Validation 예측 결과 재사용

import numpy as np
import pandas as pd

# 기존 모델 A 기준 점수
baseline_f5 = best_a[0]
baseline_th = best_a[1]

# 0.01 ~ 0.20 구간을 0.005 간격으로 탐색
thresholds = np.round(
    np.arange(0.01, 0.2001, 0.005),
    4
)

results = []

for th in thresholds:
    predictions = probs_to_labels(val_a, float(th))

    score, detail = f5_score(
        va_docs,
        predictions
    )

    results.append({
        "threshold": float(th),
        "F5": float(score),
        "TP": detail[0],
        "FP": detail[1],
        "FN": detail[2]
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    "F5",
    ascending=False
).reset_index(drop=True)

print("=" * 60)
print("기존 모델 A")
print("F5:", baseline_f5)
print("Threshold:", baseline_th)

print("\n정밀 탐색 TOP 10")
print(results_df.head(10).to_string(index=False))

best_row = results_df.iloc[0]

print("\n" + "=" * 60)
print("정밀 탐색 최고 F5:", best_row["F5"])
print("최적 Threshold:", best_row["threshold"])
print("TP:", int(best_row["TP"]))
print("FP:", int(best_row["FP"]))
print("FN:", int(best_row["FN"]))
print(
    "기존 대비 F5 변화:",
    round(best_row["F5"] - baseline_f5, 6)
)
print("=" * 60)

# 이후 제출 및 실험에서 사용할 값
best_threshold_a = float(best_row["threshold"])
best_f5_a = float(best_row["F5"])


기존 모델 A
F5: 0.9150943396226415
Threshold: 0.08

정밀 탐색 TOP 10
 threshold       F5  TP  FP  FN
     0.025 0.923497 494 339  29
     0.020 0.922237 494 358  29
     0.030 0.921216 492 319  31
     0.040 0.921134 491 293  32
     0.035 0.920404 491 304  32
     0.015 0.919728 494 396  29
     0.070 0.917802 487 234  36
     0.065 0.917337 487 241  36
     0.050 0.917227 488 270  35
     0.010 0.917167 494 435  29

정밀 탐색 최고 F5: 0.9234972677595629
최적 Threshold: 0.025
TP: 494
FP: 339
FN: 29
기존 대비 F5 변화: 0.008403


- Validation F5가 0.9151-> 0.9235로 증가

- Threshold 0.025를 새로운 기준으로 저

# 13. 개선된 모델 A 설정 저장

In [ ]:
import json

best_config = {
    "model": "microsoft/deberta-v3-base",
    "model_path": "/content/pii_deberta_base",
    "threshold": float(best_threshold_a),
    "validation_f5": float(best_f5_a),
    "repeat_name": False
}

with open(
    "/content/pii_best_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(best_config, f, indent=2)

print("최적 설정 저장 완료!")
print(json.dumps(best_config, indent=2))


최적 설정 저장 완료!
{
  "model": "microsoft/deberta-v3-base",
  "model_path": "/content/pii_deberta_base",
  "threshold": 0.025,
  "validation_f5": 0.9234972677595629,
  "repeat_name": false
}


# 14. PII 예측 및 F5 평가 함수 구조 확인 (기존 코드의 작동 방식 확인)

In [ ]:
# 14. PII 예측 및 F5 평가 함수 구조 확인
# 유형별 Threshold 최적화 준비
# ============================================================

import inspect

print("===== PII 예측 함수 =====")
print(inspect.getsource(probs_to_labels))

print("\n===== F5 평가 함수 =====")
print(inspect.getsource(f5_score))

===== PII 예측 함수 =====
def probs_to_labels(probs,threshold=0.25):
    out={}
    for doc,p in probs.items():
        pos=p[:,1:]; arg=pos.argmax(axis=1)+1; best=pos.max(axis=1)
        out[doc]=[I2L[int(k)] if s>=threshold else 'O' for k,s in zip(arg,best)]
    return out


===== F5 평가 함수 =====
def f5_score(gold,pred):
    tp=fp=fn=0
    for d in gold:
        g=d['labels']; p=pred[d['document']]
        for a,b in zip(g,p):
            if a==b and a!='O':tp+=1
            elif a!=b:
                if b!='O':fp+=1
                if a!='O':fn+=1
    return 26*tp/(26*tp+25*fn+fp) if tp+fp+fn else 0.,(tp,fp,fn)



# 15. 개인정보 유형별 Threshold 최적화

- 각 유형마다 Threshold 다르게 설정해서 최적의 조합 찾기

In [ ]:
# 15. PII 유형별 Threshold 고속 최적화 (NumPy)
# 모델 재학습 없음 / 기존 val_a 재사용

import numpy as np
import pandas as pd

BASE_TH = 0.025

candidate_thresholds = np.array([
    0.005, 0.01, 0.015, 0.02, 0.025,
    0.03, 0.04, 0.05, 0.07, 0.10,
    0.15, 0.20, 0.30, 0.50
])

label_types = sorted({
    label.split("-", 1)[1]
    for label in LABELS if label != "O"
})

# 1. Validation 데이터 준비
gold_by_doc = {
    d["document"]: d["labels"]
    for d in va_docs
}

pred_ids_list = []
best_probs_list = []
gold_ids_list = []

for doc, p in val_a.items():
    pos = p[:, 1:]
    pred_ids_list.append(pos.argmax(axis=1) + 1)
    best_probs_list.append(pos.max(axis=1))

    gold_ids_list.append(
        np.array(
            [L2I[label] for label in gold_by_doc[doc]],
            dtype=np.int16
        )
    )

pred_ids = np.concatenate(pred_ids_list).astype(np.int16)
best_probs = np.concatenate(best_probs_list).astype(np.float32)
gold_ids = np.concatenate(gold_ids_list).astype(np.int16)

assert len(pred_ids) == len(gold_ids), "토큰 개수 불일치"

# 2. PII 유형별 라벨 ID
type_to_ids = {
    t: np.array([
        L2I[label]
        for label in LABELS
        if label != "O" and label.split("-", 1)[1] == t
    ])
    for t in label_types
}

# 3. 현재 Threshold 기준 평가
def fast_f5(thresholds):
    active_thresholds = np.full(
        len(LABELS), BASE_TH, dtype=np.float32
    )

    for t, th in thresholds.items():
        active_thresholds[type_to_ids[t]] = th

    predicted = np.where(
        best_probs >= active_thresholds[pred_ids],
        pred_ids,
        0
    )

    tp = int(np.count_nonzero(
        (predicted == gold_ids) & (gold_ids != 0)
    ))
    fp = int(np.count_nonzero(
        (predicted != gold_ids) & (predicted != 0)
    ))
    fn = int(np.count_nonzero(
        (predicted != gold_ids) & (gold_ids != 0)
    ))

    denom = 26 * tp + 25 * fn + fp
    f5 = 26 * tp / denom if denom else 0.0

    return f5, (tp, fp, fn)

baseline_f5, baseline_counts = fast_f5({})

print("기준 F5:", baseline_f5)
print("기준 TP, FP, FN:", baseline_counts)

# 기존 f5_score와 계산 결과 일치 확인
assert abs(baseline_f5 - best_f5_a) < 1e-8, (
    "기존 점수와 불일치! 라벨 매핑을 확인하세요."
)

# 4. 유형별 Threshold 순차 최적화
# 한 유형의 최적값을 다음 유형에도 반영하는 방식
best_thresholds_by_type = {
    t: BASE_TH for t in label_types
}

results = []

for t in label_types:
    current_best_f5, _ = fast_f5(best_thresholds_by_type)
    current_best_th = best_thresholds_by_type[t]

    for th in candidate_thresholds:
        trial = best_thresholds_by_type.copy()
        trial[t] = float(th)

        score, _ = fast_f5(trial)

        if score > current_best_f5 + 1e-12:
            current_best_f5 = score
            current_best_th = float(th)

    best_thresholds_by_type[t] = current_best_th

    results.append({
        "PII_type": t,
        "best_threshold": current_best_th,
        "F5": current_best_f5
    })

# 5. 최종 평가
final_f5, final_counts = fast_f5(
    best_thresholds_by_type
)

best_f5_by_type = final_f5

print("\n===== 유형별 Threshold 결과 =====")
print(pd.DataFrame(results).to_string(index=False))

print("\n===== 최종 결과 =====")
print("기존 F5:", baseline_f5)
print("새로운 F5:", final_f5)
print("F5 변화량:", final_f5 - baseline_f5)
print("TP, FP, FN:", final_counts)
print("유형별 Threshold:", best_thresholds_by_type)

if final_f5 > baseline_f5:
    print("\n유형별 Threshold 개선 성공!")
else:
    print("\n개선 없음. 기존 설정 유지.")

기준 F5: 0.9234972677595629
기준 TP, FP, FN: (494, 339, 29)

===== 유형별 Threshold 결과 =====
      PII_type  best_threshold       F5
         EMAIL           0.150 0.923630
        ID_NUM           0.025 0.923630
  NAME_STUDENT           0.025 0.923630
     PHONE_NUM           0.200 0.924228
STREET_ADDRESS           0.025 0.924228
  URL_PERSONAL           0.040 0.924761
      USERNAME           0.025 0.924761

===== 최종 결과 =====
기존 F5: 0.9234972677595629
새로운 F5: 0.9247606019151847
F5 변화량: 0.0012633341556218536
TP, FP, FN: (494, 320, 29)
유형별 Threshold: {'EMAIL': 0.15, 'ID_NUM': 0.025, 'NAME_STUDENT': 0.025, 'PHONE_NUM': 0.2, 'STREET_ADDRESS': 0.025, 'URL_PERSONAL': 0.04, 'USERNAME': 0.025}

유형별 Threshold 개선 성공!


#16. 최종 submission 생성 및 다운로드

In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import files

# 1. 최적 Threshold 설정
final_thresholds = best_thresholds_by_type.copy()

# 2. Test 데이터 예측
print("Test 데이터 예측 중...")

test_probs = infer(model_a, tok_a, test)

# 3. 유형별 Threshold 적용
def final_predict(probs, thresholds):
    result = {}

    for doc_id, p in probs.items():
        pos = p[:, 1:]
        pred_ids = pos.argmax(axis=1) + 1
        pred_probs = pos.max(axis=1)

        labels = []

        for label_id, prob in zip(pred_ids, pred_probs):
            label = I2L[int(label_id)]
            pii_type = label.split("-", 1)[1]
            threshold = thresholds[pii_type]

            if prob >= threshold:
                labels.append(label)
            else:
                labels.append("O")

        result[doc_id] = labels

    return result

test_predictions = final_predict(
    test_probs,
    final_thresholds
)

# 4. Kaggle Submission 형식
rows = []
row_id = 0

for doc in test:
    doc_id = doc["document"]
    tokens = doc["tokens"]
    predictions = test_predictions[doc_id]

    for token_id, label in enumerate(predictions):
        if label != "O":
            rows.append({
                "row_id": row_id,
                "document": doc_id,
                "token": token_id,
                "label": label
            })
            row_id += 1

submission = pd.DataFrame(
    rows,
    columns=["row_id", "document", "token", "label"]
)

# 5. 저장
SAVE_FILE = "/content/submission.csv"
submission.to_csv(SAVE_FILE, index=False)

print("\nSubmission 저장 완료!")
print("저장 위치:", SAVE_FILE)
print("예측 PII 개수:", len(submission))
print(submission.head(10))

# 6. 다운로드
files.download(SAVE_FILE)


Test 데이터 예측 중...

Submission 저장 완료!
저장 위치: /content/submission.csv
예측 PII 개수: 27
   row_id  document  token           label
0       0         7      9  B-NAME_STUDENT
1       1         7     10  I-NAME_STUDENT
2       2         7    482  B-NAME_STUDENT
3       3         7    483  I-NAME_STUDENT
4       4         7    741  B-NAME_STUDENT
5       5         7    742  I-NAME_STUDENT
6       6        10      0  B-NAME_STUDENT
7       7        10      1  I-NAME_STUDENT
8       8        10    464  B-NAME_STUDENT
9       9        10    465  I-NAME_STUDENT


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 19. 학습 완료된 모델 A ZIP 백업 및 다운로드

In [ ]:
import os
import shutil
from google.colab import files

MODEL_PATH = "/content/pii_deberta_base"

# 저장된 모델이 존재하는지 확인
assert os.path.isdir(MODEL_PATH), (
    "모델 폴더가 없습니다. Colab 세션 또는 저장 경로를 확인하세요."
)

print("===== 저장된 모델 파일 =====")
for filename in os.listdir(MODEL_PATH):
    path = os.path.join(MODEL_PATH, filename)
    print(filename, round(os.path.getsize(path) / 1024**2, 2), "MB")

# ZIP 압축
zip_path = shutil.make_archive(
    "/content/pii_deberta_base",
    "zip",
    root_dir="/content",
    base_dir="pii_deberta_base"
)

print("\nZIP 저장 완료:", zip_path)
print("ZIP 크기:", round(os.path.getsize(zip_path) / 1024**2, 2), "MB")

# 컴퓨터로 다운로드
files.download(zip_path)

===== 저장된 모델 파일 =====
model.safetensors 701.32 MB
config.json 0.0 MB
tokenizer.json 7.95 MB
tokenizer_config.json 0.0 MB

ZIP 저장 완료: /content/pii_deberta_base.zip
ZIP 크기: 589.82 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>